## LIBRERIA

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify 
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

## 1.CONEXIÓN BASE DE DATOS

In [3]:
DB_HOST = 'localhost'
DB_PORT = 5432
DB_NAME = 'censo_2024'
DB_USER = 'postgres'
DB_PASSWORD = 'postgres' 

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

## 2.CARGAR DATOS

In [4]:
sql_comunas = '''
SELECT nom_comuna, geom
FROM dpa.comunas_rm_shp;
'''

gdf_comunas    = gpd.read_postgis(sql_comunas, engine, geom_col='geom')
gdf_centroides = gdf_comunas.copy()
gdf_centroides['geometry'] = gdf_comunas.centroid

print(f'Comunas: {len(gdf_comunas)}')
display(gdf_comunas)

Comunas: 52


,nom_comuna,geom
0,LO PRADO,"MULTIPOLYGON (((339891.406 6298725.5, 339967.9..."
1,INDEPENDENCIA,"MULTIPOLYGON (((345568.713 6303028.6, 345794.7..."
2,LO ESPEJO,"MULTIPOLYGON (((344800.111 6290835.676, 343521..."
3,SAN RAMÓN,"MULTIPOLYGON (((348001.245 6290000.561, 348259..."
4,LA CISTERNA,"MULTIPOLYGON (((346490 6286649.498, 345256.375..."
5,PEDRO AGUIRRE CERDA,"MULTIPOLYGON (((344800.111 6290835.676, 342530..."
6,SAN MIGUEL,"MULTIPOLYGON (((348011.801 6290001.611, 348001..."
7,CONCHALÍ,"MULTIPOLYGON (((342803.041 6306876.533, 343470..."
8,SAN JOAQUÍN,"MULTIPOLYGON (((349191.057 6294882.314, 349437..."
9,LA GRANJA,"MULTIPOLYGON (((350573.769 6287953.638, 350492..."


## 3.INDICADORES

In [12]:
sql_indicadores = '''
WITH agg AS 
(
SELECT c.nom_comuna, 
z.geocodigo::DOUBLE PRECISION AS geocodigo, 
ROUND (((COUNT(*) FILTER (WHERE p.p15>=12 and p.p15<=14))*100.0/COUNT(*)),2) AS ptje_esc_mayor_16,
ROUND (((COUNT(*) FILTER (WHERE p.p12pais<>998))*100.0/COUNT(*)),2) AS ptje_migrantes
FROM public.personas AS p
JOIN public.hogares AS h
ON p.hogar_ref_id = h.hogar_ref_id
JOIN public.viviendas AS v
ON h.vivienda_ref_id = v.vivienda_ref_id
JOIN public.zonas AS z
ON v.zonaloc_ref_id = z.zonaloc_ref_id
JOIN public.comunas AS c
ON z.codigo_comuna = c.codigo_comuna
GROUP BY c.nom_comuna, z.geocodigo
)
SELECT a.*, shp.geom
FROM agg AS a
JOIN dpa.comunas_rm_shp AS shp
ON shp.geocodigo = a.geocodigo;
'''

In [13]:
gdf = gpd.read_postgis(sql_indicadores, engine, geom_col='geom')

DatabaseError: Execution failed on sql '
WITH agg AS 
(
SELECT c.nom_comuna, 
z.geocodigo::DOUBLE PRECISION AS geocodigo, 
ROUND (((COUNT(*) FILTER (WHERE p.p15>=12 and p.p15<=14))*100.0/COUNT(*)),2) AS ptje_esc_mayor_16,
ROUND (((COUNT(*) FILTER (WHERE p.p12pais<>998))*100.0/COUNT(*)),2) AS ptje_migrantes
FROM public.personas AS p
JOIN public.hogares AS h
ON p.hogar_ref_id = h.hogar_ref_id
JOIN public.viviendas AS v
ON h.vivienda_ref_id = v.vivienda_ref_id
JOIN public.zonas AS z
ON v.zonaloc_ref_id = z.zonaloc_ref_id
JOIN public.comunas AS c
ON z.codigo_comuna = c.codigo_comuna
GROUP BY c.nom_comuna, z.geocodigo
)
SELECT a.*, shp.geom
FROM agg AS a
JOIN dpa.comunas_rm_shp AS shp
ON shp.geocodigo = a.geocodigo;
': (psycopg2.errors.UndefinedColumn) no existe la columna shp.geocodigo
LINE 22: ON shp.geocodigo = a.geocodigo;
            ^
HINT:  Probablemente quiera hacer referencia a la columna «a.geocodigo».

[SQL: 
WITH agg AS 
(
SELECT c.nom_comuna, 
z.geocodigo::DOUBLE PRECISION AS g